# Spiking neural network. Internal noise

## Required libraries

In [20]:
import snntorch as snn
from snntorch import spikeplot as splt
from snntorch import spikegen
from snntorch import surrogate
from snntorch import utils
from snntorch import functional as SF

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
import numpy as np
import itertools
import os
import shutil

## Data loading

In [21]:
# Dataloader arguments
batch_size = 128
data_path='/tmp/data/mnist'
sequence_len = 200

dtype = torch.float
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
# Define a transform
transform = transforms.Compose([
            transforms.Resize((28, 28)),
            transforms.Grayscale(),
            transforms.ToTensor(),
            transforms.Normalize((0,), (1,))])

mnist_train = datasets.MNIST(data_path, train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(data_path, train=False, download=True, transform=transform)
# Create DataLoaders
train_loader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(mnist_test, batch_size=batch_size, shuffle=True, drop_last=True)

train_loader_many = DataLoader(mnist_train, batch_size=1000, shuffle=True, drop_last=True)
test_loader_many = DataLoader(mnist_test, batch_size=1000, shuffle=True, drop_last=True)


## Additional functions

In [22]:
def forward_pass(net, data):
 spk_rec = []
 mem_rec = []
 utils.reset(net)  # resets hidden states for all LIF neurons in net

 for step in range(data.size(0)):  # data.size(0) = number of time steps
  spk_out, mem_out = net(data[step])
  spk_rec.append(spk_out)
  mem_rec.append(mem_out)

 return torch.stack(spk_rec)  # , torch.stack(mem_rec)

def calc_accuracy_for_all_data (net, train_loader_many, batch_len, sequence_len) :
    spk_rec_all = []
    training_acc = 0
    training_loss = 0
    k=0
    t = torch.empty(sequence_len, batch_len, 784)
    for i, (data, targets) in enumerate(iter(train_loader_many)):
        for ii in range(sequence_len):
            t[ii, :, :] = data.view(batch_len, -1)
        data = t.to(device)
        targets = targets.to(device)

        spk_rec = forward_pass(net, data)
        loss_val = loss_fn(spk_rec, targets).item()
        acc = SF.accuracy_rate(spk_rec, targets)
        training_acc += acc
        training_loss += loss_val
        k += 1
        
    training_acc = training_acc / k
    training_loss = training_loss / k
    del data, targets, t

    return training_acc, training_loss


## Custom noisy SNN

In [29]:
class CustomNet(nn.Module): 
    def __init__(self):
        super().__init__()

        # Initialize layers
        self.fc1 = nn.Linear(784, neurons_hidd, bias=False)
        self.filt = nn.Tanh() 
        if filter_type == 'sigm' : self.filt = nn.Sigmoid()
        
        self.lif1 = snn.Leaky(beta=beta, threshold=1.0, spike_grad=spike_grad, reset_mechanism=reset_mech)
        self.fc2 = nn.Linear(neurons_hidd, 10, bias=False)
        self.lif2 = snn.Leaky(beta=beta, threshold=1.0, spike_grad=spike_grad, reset_mechanism=reset_mech)

    def load_states(self, another_net) :
        print(len(another_net.state_dict().keys()), another_net.state_dict().keys())
        print(len(self.state_dict().keys()), self.state_dict().keys())
        if self.state_dict().keys() == another_net.state_dict().keys():
            self.load_state_dict(another_net.state_dict())
        else:
            print("Warning! Network architectures are not same! Should be checked!")
            
        dict_copy = another_net.state_dict()
        dict_this = self.state_dict()
        dict_keys_copy = list(dict_copy.keys())
        dict_keys_this = list(dict_this.keys())

        if len(dict_this) == len(dict_copy) :
            for i in range(len(dict_this)) : 
                dict_this[dict_keys_this[i]] = dict_copy[dict_keys_copy[i]].clone()
        else :
            print("Error! The expected number of parameters is not equal to the set one!")
        self.load_state_dict(dict_this)

    def forward(self, x):

        # Initialize hidden states at t=0
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()

        # Record
        spk1_rec = []
        mem1_rec = []
        spk2_rec = []
        mem2_rec = []

        for step in range(x.size(0)):
            if filter_type == 'none' : cur1 = self.fc1(x[step,:].reshape(x.size(1), 784))
            else : cur1 = self.filt(self.fc1(x[step,:].reshape(x.size(1), 784)))
            spk1, mem1 = self.lif1(cur1, mem1)
            #cur2 = self.filt(self.fc2(spk1))
            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)

            spk1_rec.append(spk1)
            mem1_rec.append(mem1)
            spk2_rec.append(spk2)
            mem2_rec.append(mem2)

        return torch.stack(spk2_rec, dim=0), torch.stack(mem2_rec, dim=0), torch.stack(spk1_rec, dim=0), torch.stack(mem1_rec, dim=0)

    def forward_noise(self, x, D):

        # Initialize hidden states at t=0
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()

        # Record
        spk1_rec = []
        mem1_rec = []
        spk2_rec = []
        mem2_rec = []

        for step in range(x.size(0)):
            # Generate input current for hidden layer
            if filter_type == 'none' : cur1 = self.fc1(x[step,:].reshape(x.size(1), 784))
            else : cur1 = self.filt(self.fc1(x[step,:].reshape(x.size(1), 784)))
            
            # Add noise to the input current if needed
            if noise_to == 'cur1' :
                if noise_comm : noise = np.dot(np.sqrt(2*D) * np.random.normal(0, 1, size=(cur1.size(0),1)), np.ones((1,cur1.size(1)))).astype(np.float32)
                else : noise = np.sqrt(2*D) * np.random.normal(0, 1, size=(cur1.size(0),cur1.size(1))).astype(np.float32)
                if noise_type == 'A' : cur1 = torch.from_numpy(cur1.detach().cpu().numpy() + noise)
                if noise_type == 'M' : cur1 = torch.from_numpy(cur1.detach().cpu().numpy()*(1+noise))
                cur1 = cur1.to(device)

            # Generate output signal of hidden layer and calculate the membrane potential
            spk1, mem1 = self.lif1(cur1, mem1)

            # Add noise to the membrane potential if needed
            if noise_to == 'mem1' :
                if noise_comm : noise = np.dot(np.sqrt(2*D) * np.random.normal(0, 1, size=(mem1.size(0),1)), np.ones((1,mem1.size(1)))).astype(np.float32)
                else : noise = np.sqrt(2*D) * np.random.normal(0, 1, size=(mem1.size(0),mem1.size(1))).astype(np.float32)
                if noise_type == 'A' : mem1 = torch.from_numpy(mem1.detach().cpu().numpy() + noise)
                if noise_type == 'M' : mem1 = torch.from_numpy(mem1.detach().cpu().numpy()*(1+noise))
                mem1 = mem1.to(device)

            # Add noise to the output signal of hidden layer if needed
            if noise_to == 'spk1' :
                if noise_comm : noise = np.dot(np.sqrt(2*D) * np.random.normal(0, 1, size=(spk1.size(0),1)), np.ones((1,spk1.size(1)))).astype(np.float32)
                else : noise = np.sqrt(2*D) * np.random.normal(0, 1, size=(spk1.size(0),spk1.size(1))).astype(np.float32)
                if noise_type == 'A' : spk1 = torch.from_numpy(spk1.detach().cpu().numpy() + noise)
                if noise_type == 'M' : spk1 = torch.from_numpy(spk1.detach().cpu().numpy()*(1+noise))
                spk1 = spk1.to(device)
            
            # Generate input signal of output layer
            #cur2 = self.filt(self.fc2(spk1)) # if filter is applied to the output layer, too
            cur2 = self.fc2(spk1)
            # Generate output signal of output layer and calculate the membrane potential
            spk2, mem2 = self.lif2(cur2, mem2)

            spk1_rec.append(spk1)
            mem1_rec.append(mem1)
            spk2_rec.append(spk2)
            mem2_rec.append(mem2)

        return torch.stack(spk2_rec, dim=0), torch.stack(mem2_rec, dim=0), torch.stack(spk1_rec, dim=0), torch.stack(mem1_rec, dim=0)


    def custom_accuracy_for_all_data (self, train_loader_many, batch_len, sequence_len) :
    	spk_rec_all = []
    	training_acc = 0
    	training_loss = 0
    	k=0
    	t = torch.empty(sequence_len, batch_len, 784)
    	for i, (data, targets) in enumerate(iter(train_loader_many)):
    		for ii in range(sequence_len):
    			t[ii, :, :] = data.view(batch_len, -1)
    		data = t.to(device)
    		targets = targets.to(device)
    		
    		spk_rec, _, _, _ = self.forward(data)
    		loss_val = loss_fn(spk_rec, targets).item()
    		acc = SF.accuracy_rate(spk_rec, targets)
    		training_acc += acc
    		training_loss += loss_val
    		k += 1
    	
    	training_acc = training_acc / k
    	training_loss = training_loss / k
    	del data, targets, t
    	return training_acc, training_loss

    def custom_accuracy_for_all_data_noise (self, train_loader_many, batch_len, sequence_len, D) :
    	spk_rec_all = []
    	training_acc = 0
    	training_loss = 0
    	k=0
    	t = torch.empty(sequence_len, batch_len, 784)
    	for i, (data, targets) in enumerate(iter(train_loader_many)):
    		for ii in range(sequence_len):
    			t[ii, :, :] = data.view(batch_len, -1)
    		data = t.to(device)
    		targets = targets.to(device)
    		
    		spk_rec, _, _, _ = self.forward_noise(data, D=D)
    		
    		loss_val = loss_fn(spk_rec, targets).item()
    		acc = SF.accuracy_rate(spk_rec, targets)
    		training_acc += acc
    		training_loss += loss_val
    		k += 1
    	
    	#print(spk_rec.size())
    	#print(k)
    	training_acc = training_acc / k
    	training_loss = training_loss / k
    	del data, targets, t
    	return training_acc, training_loss

## Main parameters

In [39]:
#network_path = 'last_net0/'
neurons_hidd=100
noise_type = 'A' # A-additive M-multiplicative
noise_comm = True  # False - Uncommon noise;     True - Common noise;
noise_comm_str = 'U'
if noise_comm : noise_comm_str = 'C'
filter_type = 'tanh' # 'sigm' or 'none'
noise_to = 'cur1' # 'cur1' 'mem1' or 'spk1'

dtype = torch.float
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

#optimizer = torch.optim.Adam(net.parameters(), lr=5e-4, betas=(0.9, 0.999))
loss_fn = SF.mse_count_loss(correct_rate=0.8, incorrect_rate=0.2)
spike_grad = surrogate.atan()
beta = 0.9
reset_mech = 'subtract'

K=20           # number of repetitions to calculate the statistics of accuracy values for each noise intensity
D_min = 0      # minimal noise intensity
D_max = 1      # maximal noise intensity
D_stp = 0.05   # noise intensity step

trained_network_path = 'last_net0'

## Define the network topology (choose the one of 3 variants)

#### without filters

In [18]:
net = nn.Sequential(nn.Flatten(),
                    nn.Linear(784, neurons_hidd, bias=False),
                    snn.Leaky(beta=beta, threshold=1.0, spike_grad=spike_grad, reset_mechanism=reset_mech, init_hidden=True),
                    nn.Linear(neurons_hidd, 10, bias=False),
                    snn.Leaky(beta=beta, threshold=1.0, spike_grad=spike_grad, reset_mechanism=reset_mech, init_hidden=True, output=True)
                    ).to(device)

#### sigmoid

In [40]:
net = nn.Sequential(nn.Flatten(),
                    nn.Linear(784, neurons_hidd, bias=False),
                    nn.Sigmoid(),
                    snn.Leaky(beta=beta, threshold=1.0, spike_grad=spike_grad, reset_mechanism=reset_mech, init_hidden=True),
                    nn.Linear(neurons_hidd, 10, bias=False),
                    snn.Leaky(beta=beta, threshold=1.0, spike_grad=spike_grad, reset_mechanism=reset_mech, init_hidden=True, output=True)
                    ).to(device)

#### tanh

In [16]:
net = nn.Sequential(nn.Flatten(),
                    nn.Linear(784, neurons_hidd, bias=False),
                    nn.Tanh(),
                    snn.Leaky(beta=beta, threshold=1.0, spike_grad=spike_grad, reset_mechanism=reset_mech, init_hidden=True),
                    nn.Linear(neurons_hidd, 10, bias=False),
                    snn.Leaky(beta=beta, threshold=1.0, spike_grad=spike_grad, reset_mechanism=reset_mech, init_hidden=True, output=True)
                    ).to(device)

## Testing with noise

In [ ]:
print('#'*40,'\n', trained_network_path, '\n', '#'*40)
net.load_state_dict(torch.load(trained_network_path+"/topology"))

acc_training, loss_training = calc_accuracy_for_all_data(net, train_loader_many, batch_len=1000, sequence_len=sequence_len)
acc_testing, loss_testing = calc_accuracy_for_all_data(net, test_loader_many, batch_len=1000, sequence_len=sequence_len)
print(acc_training, acc_testing, loss_training, loss_testing)

net_custom = CustomNet().to(device)
net_custom.load_states(net)

acc_training, loss_training = net_custom.custom_accuracy_for_all_data(train_loader_many, batch_len=1000, sequence_len=sequence_len)
acc_testing, loss_testing   = net_custom.custom_accuracy_for_all_data(test_loader_many, batch_len=1000, sequence_len=sequence_len)
print("Training acc. ", acc_training)
print("Testing acc. ", acc_testing)

array_D = []
array_acc = []

D_range = np.arange(0, D_max+D_stp, D_stp)
array_acc_all = np.zeros((len(D_range), K+1))
Di = 0
for D in D_range :
    array_acc_K = np.zeros(K)
    for i in range(K) :
        array_acc_K[i], loss_testing = net_custom.custom_accuracy_for_all_data_noise(test_loader_many, batch_len=1000, sequence_len=sequence_len, D=D)
        if i % 10 == 0 : print(i)
    array_acc_all[Di,0] = D
    array_acc_all[Di,1:] = 100*np.array(array_acc_K)
    array_D.append(D)
    array_acc.append(np.mean(array_acc_K)*100)
    print("D=", D, " test.acc. ", array_acc[-1])
    
    np.savetxt( trained_network_path + "/D"+noise_type+"_"+noise_comm_str+"_acc_"+noise_to+"_K="+str(K)+".txt", np.array([array_D, array_acc]).T, fmt='%.5f')
    np.savetxt( trained_network_path + "/D"+noise_type+"_"+noise_comm_str+"_acc_"+noise_to+"_stat.txt", np.array(array_acc_all).T, fmt='%.5f')
    Di+=1

######################################## 
 last_net0 
 ########################################
0.9597333333333334 0.9573 0.8597525884707768 0.8773570120334625
10 odict_keys(['1.weight', '3.threshold', '3.graded_spikes_factor', '3.reset_mechanism_val', '3.beta', '4.weight', '5.threshold', '5.graded_spikes_factor', '5.reset_mechanism_val', '5.beta'])
10 odict_keys(['fc1.weight', 'lif1.threshold', 'lif1.graded_spikes_factor', 'lif1.reset_mechanism_val', 'lif1.beta', 'fc2.weight', 'lif2.threshold', 'lif2.graded_spikes_factor', 'lif2.reset_mechanism_val', 'lif2.beta'])
Warning! Network architectures are not same! Should be checked!
Training acc.  0.9298499999999997
Testing acc.  0.9245000000000001
0
